In [0]:
%sql
SELECT * FROM project.bronze.product LIMIT 10;

In [0]:
%sql
-- Έλεγχος αν το product_key είναι μοναδικό 
SELECT 
    product_key, 
    COUNT(*) AS cnt
FROM project.bronze.product
GROUP BY product_key
HAVING cnt > 1;

In [0]:
%sql
-- Έλεγχος για περίεργα χρώματα ή κενά
SELECT DISTINCT color FROM project.bronze.product;

In [0]:
%sql
-- Έλεγχος για τα σύμβολα νομισμάτων στο standard_cost
SELECT DISTINCT 
    LEFT(standard_cost, 1) as start_char,
    RIGHT(standard_cost, 1) as end_char 
FROM project.bronze.product;

In [0]:
%sql
-- Έλεγχος για κενά κλειδιά (NULL values)
SELECT 
    COUNT(*) AS null_product_key_cnt
FROM project.bronze.product
WHERE product_key IS NULL;

In [0]:
from pyspark.sql import functions as F

# Load Bronze Table
df_product = spark.table("project.bronze.product")

# Universal String Cleaning ---
# Trim, remove '|', and map pseudo-nulls to actual NULLs
string_cols = [c for c, t in df_product.dtypes if t == "string"]
for c in string_cols:
    cleaned = F.trim(F.regexp_replace(F.col(c), r"\|", ""))
    df_product = df_product.withColumn(c, 
        F.when(F.upper(cleaned).isin("N/A", "NA", "NULL", "NONE", "-", ""), None)
         .otherwise(cleaned)
    )

# Standardize Colors & Impute Nulls
if "color" in df_product.columns:
    df_product = df_product.withColumn("color", F.initcap(F.lower(F.col("color"))))
    df_product = df_product.fillna("Unknown", subset=["color"])

# Split Standard Cost & Currency Detection
if "standard_cost" in df_product.columns:
    # Detect Currency (Using raw string 'r' to prevent SyntaxWarnings)
    df_product = df_product.withColumn("currency",
        F.when(F.col("standard_cost").rlike(r"\$"), "USD")
         .when(F.col("standard_cost").rlike("€"), "EUR")
         .when(F.col("standard_cost").rlike("£"), "GBP")
         .otherwise("USD") # Defaulting to USD if no symbol is found
    )
    # Clean the string and cast directly to Decimal(18,2) for financial accuracy
    df_product = df_product.withColumn("standard_cost", 
        F.regexp_replace(F.col("standard_cost"), r"[$,€£\s]", "").cast("decimal(18,2)")
    )

# Remove null keys and duplicate IDs (Crucial for Dimension Tables)
df_product = df_product.filter(F.col("product_key").isNotNull())
df_product = df_product.dropDuplicates(["product_key"])

# Drop Bronze metadata and add Silver ingestion timestamp
df_product = df_product.drop("source_file", "ingestion_timestamp")
df_product = df_product.withColumn("ingestion_timestamp", F.current_timestamp())

# 7. Write to Silver
(df_product.write
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .format("delta")
            .saveAsTable("project.silver.product")
)

# Sanity Check εμφάνιση
display(spark.table("project.silver.product"))


In [0]:
%sql
SELECT * FROM project.bronze.sales_2017 LIMIT 5;

In [0]:
%sql
SELECT * FROM project.bronze.sales_2018 LIMIT 5;

In [0]:
%sql
SELECT * FROM project.bronze.sales_2019 LIMIT 5;

In [0]:
%sql
SELECT * FROM project.bronze.sales_2020 LIMIT 5;

In [0]:
%sql
--Schema Check if discount column exists before we add it
DESCRIBE project.bronze.sales_2020;

In [0]:
#Sabotage
from pyspark.sql import functions as F

df_2020 = spark.table("project.bronze.sales_2020")

# Προσθήκη discount, αλλαγή ημερομηνίας σε ISO, και null σε ένα κλειδί
df_2020 = df_2020.withColumn("discount", F.when(F.col("quantity") > 5, F.lit("$5.00")).otherwise(F.lit(None))) \
                 .withColumn("order_date", F.when(F.col("sales_order_number") == "SO63283", F.lit("2020-05-15")).otherwise(F.col("order_date"))) \
                 .withColumn("product_key", F.when(F.col("sales_order_number") == "SO63283", F.lit(None)).otherwise(F.col("product_key")))

df_2020.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("project.bronze.sales_2020")
print("Sabotage Complete.")

In [0]:
%sql
--Sabotage check
SELECT sales_order_number, order_date, product_key, discount 
FROM project.bronze.sales_2020 
WHERE sales_order_number = 'SO63283';

In [0]:
%sql
-----delete ?
SELECT sales_order_number, quantity, discount 
FROM project.bronze.sales_2020 
WHERE discount IS NOT NULL 
LIMIT 5;

In [0]:
%sql
-- Έλεγχος για NULL κλειδιά στα Bronze δεδομένα (πριν τον καθαρισμό)
-- Θα πρέπει να δούμε την παραγγελία SO63283 που κάναμε σαμποτάζ
SELECT '2020' as source, sales_order_number, product_key, reseller_key, employee_key
FROM project.bronze.sales_2020
WHERE sales_order_number IS NULL 
   OR product_key IS NULL 
   OR reseller_key IS NULL 
   OR employee_key IS NULL;

In [0]:
%sql
-- Έλεγχος αν ο συνδυασμός Order Number και Product Key είναι μοναδικός
-- Αν το αποτέλεσμα είναι κενό, τότε δεν έχουμε διπλότυπα line items
SELECT 
    sales_order_number, 
    product_key, 
    COUNT(*) as cnt
FROM project.bronze.sales_2020
GROUP BY sales_order_number, product_key
HAVING COUNT(*) > 1;

In [0]:
%sql
-- Σύγκριση συνολικών γραμμών vs μοναδικών παραγγελιών
SELECT 
    COUNT(*) AS total_rows,
    COUNT(DISTINCT sales_order_number) AS distinct_orders
FROM project.bronze.sales_2020;

In [0]:
%sql
-- Σύγκριση συνολικών γραμμών vs μοναδικών εγγραφών (Primary Key Check)
-- Αν τα νούμερα είναι ίδια, δεν έχουμε διπλότυπα
SELECT 
    COUNT(*) AS total_rows,
    COUNT(DISTINCT sales_order_number, product_key) AS distinct_pk_cnt
FROM project.bronze.sales_2020;

In [0]:
%sql
-- Έλεγχος για NULL τιμές που θα "έσπαγαν" τα Join στο Gold layer
SELECT 
    COUNT(*) - COUNT(sales_order_number) AS null_order_num,
    COUNT(*) - COUNT(product_key) AS null_product_key,
    COUNT(*) - COUNT(reseller_key) AS null_reseller_key,
    COUNT(*) - COUNT(employee_key) AS null_employee_key
FROM project.bronze.sales_2020;

In [0]:
from pyspark.sql import functions as F

# Union και καθαρισμός
s17 = spark.table("project.bronze.sales_2017")
s18 = spark.table("project.bronze.sales_2018")
s19 = spark.table("project.bronze.sales_2019")
s20 = spark.table("project.bronze.sales_2020")

df_sales = s17.unionByName(s18, allowMissingColumns=True) \
              .unionByName(s19, allowMissingColumns=True) \
              .unionByName(s20, allowMissingColumns=True)

# Διόρθωση ημερομηνιών, νομισμάτων και NULL τιμών
df_sales = df_sales.withColumn("order_date", 
    F.when(F.col("order_date").rlike(r"^\d{4}-\d{2}-\d{2}$"), F.to_date(F.col("order_date"), "yyyy-MM-dd"))
     .otherwise(F.to_date(F.regexp_replace(F.col("order_date"), r"^[A-Za-z]+,\s*", ""), "MMMM d, yyyy"))) \
    .withColumn("sales_amount", F.regexp_replace(F.col("sales"), r"[$,\s]", "").cast("decimal(18,2)")) \
    .fillna(0, subset=["discount"]) \
    .fillna(-1, subset=["product_key"])

df_sales.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("project.silver.sales")
print("✅ Silver Sales Complete.")

In [0]:
%sql
--  Έλεγχος αν το πλήθος των εγγραφών στον Silver είναι ίσο με το άθροισμα των Bronze
-- (Πρέπει το σύνολο του Silver να είναι ίσο ή ελαφρώς μικρότερο αν υπήρχαν διπλότυπα)
SELECT 
    (SELECT COUNT(*) FROM project.silver.sales) AS silver_cnt,
    (SELECT 
        (SELECT COUNT(*) FROM project.bronze.sales_2017) + 
        (SELECT COUNT(*) FROM project.bronze.sales_2018) + 
        (SELECT COUNT(*) FROM project.bronze.sales_2019) + 
        (SELECT COUNT(*) FROM project.bronze.sales_2020)
    ) AS bronze_total_cnt;


In [0]:
%sql
-- Έλεγχος για Null τιμές σε κρίσιμα πεδία
-- Το αποτέλεσμα πρέπει να είναι 0 για όλα, εκτός αν κάποιο κλειδί επιτρέπεται να είναι NULL
SELECT 
    COUNT(*) - COUNT(sales_order_number) AS null_order_num,
    COUNT(*) - COUNT(order_date) AS null_date,
    COUNT(*) - COUNT(product_key) AS null_prod_key,
    COUNT(*) - COUNT(sales_amount) AS null_sales
FROM project.silver.sales;

In [0]:
%sql
--Έλεγχος αν το product_key = -1 (από το Sabotage) υπάρχει στον Silver
SELECT COUNT(*) AS sabotaged_rows_fixed
FROM project.silver.sales
WHERE product_key = -1;